# Notebook 02 — NDVI Satelital sobre la Sierra Nevada de Santa Marta
## Teledetección — Maestría en Ingeniería, Universidad del Magdalena
**Sesión 2 · Sábado 18 de julio de 2026**

---

### ¿Qué es el NDVI?

El **NDVI (Normalized Difference Vegetation Index)** es el índice más usado en teledetección.
Mide el **vigor de la vegetación** basándose en cómo las plantas interactúan con la luz:

- La vegetación **absorbe** la luz **roja** para la fotosíntesis
- La vegetación **refleja** mucho en el **infrarrojo cercano (NIR)** por su estructura celular

$$\text{NDVI} = \frac{\text{NIR} - \text{Rojo}}{\text{NIR} + \text{Rojo}} = \frac{\text{B8} - \text{B4}}{\text{B8} + \text{B4}}$$

**Rango de valores:** -1 a +1

| NDVI | Interpretación |
|------|----------------|
| < 0 | Agua, nieve, nubes |
| 0.0 – 0.1 | Suelo desnudo, roca, zona urbana |
| 0.1 – 0.2 | Vegetación muy escasa o estresada |
| 0.2 – 0.4 | Vegetación moderada (pasturas, cultivos jóvenes) |
| 0.4 – 0.6 | Vegetación densa (cultivos en desarrollo) |
| 0.6 – 0.9 | Vegetación muy densa y sana (bosque, cacao adulto) |

---
Este notebook tiene **dos partes**:
- **Parte A:** NDVI con archivos locales (`rasterio`) — para entender el cálculo pixel a pixel
- **Parte B:** NDVI con GEE (`ee` + `geemap`) — el método que usarás en el curso

---

## PARTE A — NDVI con archivos locales (rasterio)

Usamos las bandas B4 y B8 que vienen en el repositorio (`datos/B04_10m.jp2` y `datos/B08_10m.jp2`).
Esta parte muestra el cálculo *desde cero* — cómo funciona internamente.

**Contexto:** Estas son bandas de una imagen Sentinel-2 L2A de la zona norte del Magdalena.

In [ ]:
# Instalar rasterio si no está disponible
!pip install rasterio --quiet

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import rasterio
from rasterio.plot import show

print("✓ Librerías importadas")

In [ ]:
# Descargar los datos de ejemplo del repositorio
# (solo si estás en Colab — si tienes el repo clonado localmente, ajusta las rutas)
import os

base_url = 'https://raw.githubusercontent.com/miguepoloc/teledeteccion-maestria/main/datos/'

if not os.path.exists('B04_10m.jp2'):
    !wget -q {base_url}B04_10m.jp2 -O B04_10m.jp2
    !wget -q {base_url}B08_10m.jp2 -O B08_10m.jp2
    print("✓ Bandas descargadas")
else:
    print("✓ Bandas ya disponibles localmente")

In [ ]:
# ============================================================
# CARGAR LAS BANDAS CON RASTERIO
# ============================================================

# Abrir la banda B4 (Rojo, 665 nm)
with rasterio.open('B04_10m.jp2') as src_rojo:
    banda_roja = src_rojo.read(1).astype('float32')  # leer banda 1 como float
    perfil = src_rojo.profile                         # guardar metadatos para exportar
    crs = src_rojo.crs                               # sistema de coordenadas
    transform = src_rojo.transform

# Abrir la banda B8 (NIR, 842 nm)
with rasterio.open('B08_10m.jp2') as src_nir:
    banda_nir = src_nir.read(1).astype('float32')

print("Dimensiones de la imagen:", banda_roja.shape)
print("Píxeles totales:", f"{banda_roja.size:,}")
print("Resolución: 10 m por píxel")
print("Área cubierta:", f"{banda_roja.shape[0]*10/1000:.1f} × {banda_roja.shape[1]*10/1000:.1f} km")
print("Sistema de referencia:", crs)
print("\nEstadísticas B4 (Rojo):")
print(f"  Mínimo: {banda_roja.min():.0f} | Máximo: {banda_roja.max():.0f} | Media: {banda_roja.mean():.0f}")
print("\nEstadísticas B8 (NIR):")
print(f"  Mínimo: {banda_nir.min():.0f} | Máximo: {banda_nir.max():.0f} | Media: {banda_nir.mean():.0f}")

In [ ]:
# ============================================================
# VISUALIZAR LAS BANDAS INDIVIDUALES
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Banda B4 (Rojo)
im1 = axes[0].imshow(banda_roja, cmap='Reds', vmin=0, vmax=3000)
axes[0].set_title('Banda B4 — Rojo (665 nm)\nLa vegetación aparece OSCURA\n(la clorofila absorbe la luz roja)', fontsize=11)
axes[0].axis('off')
plt.colorbar(im1, ax=axes[0], shrink=0.8, label='Reflectancia × 10000')

# Banda B8 (NIR)
im2 = axes[1].imshow(banda_nir, cmap='Greens', vmin=0, vmax=5000)
axes[1].set_title('Banda B8 — NIR (842 nm)\nLa vegetación aparece BRILLANTE\n(la estructura celular refleja el NIR)', fontsize=11)
axes[1].axis('off')
plt.colorbar(im2, ax=axes[1], shrink=0.8, label='Reflectancia × 10000')

plt.suptitle('Sentinel-2 L2A — Norte del Magdalena\n(La diferencia entre estas dos bandas es el fundamento del NDVI)', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CALCULAR EL NDVI
# ============================================================

# Evitar división por cero añadiendo un valor muy pequeño (epsilon)
# Sin esto, si NIR + Rojo = 0, Python daría error o NaN
epsilon = 1e-6

ndvi = (banda_nir - banda_roja) / (banda_nir + banda_roja + epsilon)

# Limitar al rango válido [-1, 1]
ndvi = np.clip(ndvi, -1, 1)

print("NDVI calculado exitosamente")
print(f"Dimensiones: {ndvi.shape}")
print(f"Valor mínimo: {ndvi.min():.4f}")
print(f"Valor máximo: {ndvi.max():.4f}")
print(f"Valor promedio: {ndvi.mean():.4f}")
print(f"Desv. estándar: {ndvi.std():.4f}")
print()

# Distribución por categoría
total = ndvi.size
categorias = [
    ('Agua / sin vegetación (NDVI < 0)',            ndvi < 0),
    ('Suelo desnudo (0.0 – 0.1)',                   (ndvi >= 0)   & (ndvi < 0.1)),
    ('Vegetación escasa (0.1 – 0.2)',               (ndvi >= 0.1) & (ndvi < 0.2)),
    ('Vegetación moderada (0.2 – 0.4)',             (ndvi >= 0.2) & (ndvi < 0.4)),
    ('Vegetación densa — cultivos (0.4 – 0.6)',     (ndvi >= 0.4) & (ndvi < 0.6)),
    ('Vegetación muy densa — bosque/cacao (> 0.6)', ndvi >= 0.6),
]

for nombre, mascara in categorias:
    n = np.sum(mascara)
    pct = 100 * n / total
    print(f"  {nombre}: {n:>8,} píxeles ({pct:.1f}%)")

In [ ]:
# ============================================================
# VISUALIZAR EL NDVI CON PALETA DE COLORES ESTÁNDAR
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# --- Mapa NDVI con paleta RdYlGn ---
# Rojo = valores bajos (sin vegetación), Verde = valores altos (vegetación densa)
im = axes[0].imshow(ndvi, cmap='RdYlGn', vmin=-0.1, vmax=0.9)
axes[0].set_title('NDVI — Sentinel-2 L2A\nNorte del Magdalena', fontsize=12)
axes[0].axis('off')
cbar = plt.colorbar(im, ax=axes[0], shrink=0.8)
cbar.set_label('NDVI', fontsize=10)
cbar.set_ticks([-0.1, 0, 0.1, 0.2, 0.4, 0.6, 0.9])
cbar.set_ticklabels(['< 0\nAgua', '0.0\nSuelo', '0.1', '0.2\nVeg.\nescasa', '0.4\nVeg.\nmoderada', '0.6\nVeg.\ndensa', '0.9\nBosque'])

# --- Histograma del NDVI ---
ndvi_flat = ndvi.flatten()
axes[1].hist(ndvi_flat, bins=80, color='forestgreen', edgecolor='white', alpha=0.8, density=True)
axes[1].axvline(0,   color='blue',   ls='--', lw=1.5, label='NDVI = 0 (suelo)')
axes[1].axvline(0.4, color='orange', ls='--', lw=1.5, label='NDVI = 0.4 (cultivo)')
axes[1].axvline(0.6, color='red',    ls='--', lw=1.5, label='NDVI = 0.6 (bosque/cacao)')
axes[1].set_xlabel('NDVI', fontsize=12)
axes[1].set_ylabel('Densidad de píxeles', fontsize=12)
axes[1].set_title('Distribución de valores NDVI', fontsize=12)
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('ndvi_snsm.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Imagen guardada como 'ndvi_snsm.png'")

In [ ]:
# ============================================================
# EXPORTAR EL NDVI COMO GEOTIFF
# ============================================================

# Actualizar el perfil de metadatos para el archivo de salida
perfil.update(
    dtype=rasterio.float32,
    count=1,           # una sola banda
    driver='GTiff',    # formato GeoTIFF
    nodata=-9999       # valor para píxeles sin datos
)

# Guardar el NDVI como GeoTIFF
with rasterio.open('ndvi_norte_magdalena.tif', 'w', **perfil) as dst:
    dst.write(ndvi, 1)  # escribir la banda NDVI

print("✓ NDVI exportado como 'ndvi_norte_magdalena.tif'")
print("  Este archivo puede abrirse en QGIS o SNAP para análisis adicional")

---

## PARTE B — NDVI con Google Earth Engine

Esta es la forma que usaremos en el curso. GEE carga la imagen directamente desde
la nube — no necesitas descargar nada. Y puedes aplicarlo a cualquier fecha y zona.

In [ ]:
import ee
import geemap

# Autenticar e inicializar GEE
# (si ya lo hiciste en el Notebook 01, no es necesario volver a hacerlo)
ee.Authenticate()
ee.Initialize(project='tu-proyecto-gee')

# ============================================================
# CARGAR IMAGEN SENTINEL-2 Y CALCULAR NDVI EN GEE
# ============================================================

zona_estudio = ee.Geometry.Rectangle([-74.2, 10.5, -73.8, 11.0])

def enmascarar_nubes_s2(imagen):
    """Elimina píxeles de nubes usando la banda SCL de Sentinel-2 L2A."""
    scl = imagen.select('SCL')
    mascara = scl.neq(3).and(scl.neq(8)).and(scl.neq(9)).and(scl.neq(10))
    return imagen.updateMask(mascara)

def calcular_ndvi(imagen):
    """Calcula NDVI y lo agrega como nueva banda."""
    ndvi = imagen.normalizedDifference(['B8', 'B4']).rename('NDVI')
    return imagen.addBands(ndvi)

# Cargar, limpiar y calcular NDVI
imagen_con_ndvi = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(zona_estudio)
    .filterDate('2024-01-01', '2024-03-31')
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 15))
    .map(enmascarar_nubes_s2)   # aplicar máscara de nubes a cada imagen
    .map(calcular_ndvi)         # calcular NDVI en cada imagen
    .median()                   # combinar
    .clip(zona_estudio)
)

print("✓ NDVI calculado en GEE")

# Estadísticas del NDVI
stats = imagen_con_ndvi.select('NDVI').reduceRegion(
    reducer=ee.Reducer.mean().combine(
        ee.Reducer.stdDev(), None, True
    ).combine(
        ee.Reducer.percentile([10, 25, 50, 75, 90]), None, True
    ),
    geometry=zona_estudio,
    scale=10,
    maxPixels=1e9
).getInfo()

print("\nEstadísticas NDVI — Zona Cacaotera SNSM (ene-mar 2024):")
print(f"  Promedio:        {stats.get('NDVI_mean', 0):.4f}")
print(f"  Desv. estándar: {stats.get('NDVI_stdDev', 0):.4f}")
print(f"  Mediana (p50):  {stats.get('NDVI_p50', 0):.4f}")
print(f"  Percentil 25:   {stats.get('NDVI_p25', 0):.4f}")
print(f"  Percentil 75:   {stats.get('NDVI_p75', 0):.4f}")

In [ ]:
# ============================================================
# VISUALIZAR EL NDVI EN MAPA INTERACTIVO
# ============================================================

paleta_ndvi = [
    '#d73027',  # < 0    → agua, rojo
    '#f46d43',  # 0–0.1  → suelo desnudo, naranja
    '#fdae61',  # 0.1–0.2 → muy escasa, amarillo-naranja
    '#ffffbf',  # 0.2–0.3 → escasa, amarillo
    '#d9ef8b',  # 0.3–0.4 → moderada, verde claro
    '#a6d96a',  # 0.4–0.6 → densa, verde
    '#1a9641'   # > 0.6  → muy densa (bosque, cacao), verde oscuro
]

mapa = geemap.Map()
mapa.centerObject(zona_estudio, zoom=11)

# Color natural como referencia
mapa.addLayer(
    imagen_con_ndvi,
    {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000, 'gamma': 1.4},
    'Color Natural'
)

# NDVI con paleta
mapa.addLayer(
    imagen_con_ndvi.select('NDVI'),
    {'min': -0.1, 'max': 0.9, 'palette': paleta_ndvi},
    'NDVI (ene-mar 2024)'
)

# Agregar leyenda
mapa.add_colorbar(
    {'min': -0.1, 'max': 0.9, 'palette': paleta_ndvi},
    label='NDVI'
)

mapa

**Explora el mapa:**
- Activa/desactiva las capas con los controles de Layers
- Usa el Inspector (ícono de dedo) para hacer click en cualquier punto y ver el NDVI exacto
- Identifica: ¿dónde está el NDVI más alto? ¿coincide con las zonas boscosas de la SNSM?

---

## ✅ Resumen — Diferencia entre Parte A y Parte B

| | Parte A (rasterio local) | Parte B (GEE) |
|--|--------------------------|---------------|
| **Datos** | Archivos JP2/TIF en tu computador | En la nube de Google |
| **Descarga necesaria** | Sí (~600 MB por imagen) | No |
| **Velocidad** | Depende de tu computador | Servidores de Google |
| **Área de análisis** | Limitada al archivo descargado | Global |
| **Fechas disponibles** | Solo las que descargaste | Desde 2015 hasta hoy |
| **Exportar** | GeoTIFF directo | A Google Drive |

**Para el curso usamos GEE** por practicidad. La Parte A es útil para entender el cálculo internamente.

---

## ➡️ Siguiente: Notebook 03 — NDVI con Shapefiles

Ahora que sabemos calcular NDVI, aprenderemos a recortarlo a una zona
específica definida por un polígono (shapefile) — la zona cacaotera de la SNSM.